Realizar una copia de la notebook y del dataset en el Drive personal. Referenciar al path personal. Acá les dejamos el colab usado en la clase teórica https://colab.research.google.com/drive/1IwKMezmUww_xcP07GxF6gSxjh0l3Jm9u?authuser=5#scrollTo=MXm5KTrOZ2N0


## Parte 1 - EDA

### Importo Librerias

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import optuna

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate, learning_curve

from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import (roc_auc_score, average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
                             PrecisionRecallDisplay, accuracy_score, f1_score, recall_score)


pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid')


c:\Users\Alam\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Cargo el dataset

In [2]:
RUTA = "Dataset de producción.csv"
df = pd.read_csv(RUTA)
df.head()

C:\Users\Alam\AppData\Local\Temp\ipykernel_11896\728768229.py:2: DtypeWarning: Columns (0: Observaciones) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RUTA)


,Unnamed: 0,Empresa,Año,Cuenca,Provincia,Área,Yacimiento,ID Pozo,Form.Prod.,Cód.Propio,Prod.Men.Pet.(m3),Prod.Men.Gas(Mm3),Prod.Men.Agua(m3),Prod.Acum.Pet.(m3),Prod.Acum.Gas(Mm3),Prod.Acum.Agua(m3),Iny.Agua(m3),Iny.Gas(Mm3),Iny.CO2(Mm3),Iny.Otros(m3),RGP,ETIQUETA,% de Agua,TEF,Vida Útil,Sist.Extrac.,Est.Pozo,Tipo Pozo,Clasificación,Sub clasificación,Tipo de Recurso,Sub tipo de Recurso,Observaciones,Latitud,Longitud,Cota,Profundidad,Caudal bruta
0,0,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123495,LCAB,201054500100,202.1400,4.0300,878.9500,"490,321.9480","8,455.9770","620,531.0770",0.0000,0.0000,0,0,0.0199,NaN,0.8130,31.0000,"1,356.9800",BCP,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1359,-68.7751,912.0000,"2,317.0000",34.8739
1,1,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123593,CRIN,201054501021,45.1600,0.9000,"2,996.7800","45,706.2050","1,327.2790","559,797.6500",0.0000,0.0000,0,0,0.0199,NaN,0.9852,29.9200,"1,357.3000",BES,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1817,-68.7776,924.0700,"2,330.0000",101.6691
2,2,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123598,CRIN,201054501061,138.0400,2.7400,"4,981.5900","106,582.7870","2,998.7380","923,809.1460",0.0000,0.0000,0,0,0.0198,NaN,0.9730,30.4200,"1,215.5900",BES,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1911,-68.7675,884.5800,"2,450.0000",168.2982
3,3,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123604,CRIN,201054501140,56.2100,1.1200,992.5600,"250,180.0000","7,098.9360","435,091.9350",0.0000,0.0000,0,0,0.0199,NaN,0.9464,31.0000,"1,276.4000",BM,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1822,-68.7972,964.6700,"2,429.0000",33.8313
4,4,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123612,CRIN,201054501200,233.5900,4.6400,"6,795.1200","319,410.1690","8,596.4720","1,481,590.9560",0.0000,0.0000,0,0,0.0199,NaN,0.9668,31.0000,"1,379.4500",BES,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1809,-68.7927,945.0600,"2,374.0000",226.7326


### 1.1 - Dimensiones del dataset

In [3]:
print("La cantidad de filas y columnas es:", df.shape)
df.info()


La cantidad de filas y columnas es: (174815, 38)
<class 'pandas.DataFrame'>
RangeIndex: 174815 entries, 0 to 174814
Data columns (total 38 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Unnamed: 0           174815 non-null  int64  
 1   Empresa              174815 non-null  str    
 2   Año                  174815 non-null  int64  
 3   Cuenca               174815 non-null  str    
 4   Provincia            174815 non-null  str    
 5   Área                 174815 non-null  str    
 6   Yacimiento           174815 non-null  str    
 7   ID Pozo              174815 non-null  int64  
 8   Form.Prod.           174067 non-null  str    
 9   Cód.Propio           161838 non-null  str    
 10  Prod.Men.Pet.(m3)    174815 non-null  float64
 11  Prod.Men.Gas(Mm3)    174815 non-null  float64
 12  Prod.Men.Agua(m3)    174815 non-null  float64
 13  Prod.Acum.Pet.(m3)   174815 non-null  float64
 14  Prod.Acum.Gas(Mm3)   174815 no

### 1.2 - Variables numericas y variables categoricas

In [4]:
#Se ve con df.info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 174815 entries, 0 to 174814
Data columns (total 38 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Unnamed: 0           174815 non-null  int64  
 1   Empresa              174815 non-null  str    
 2   Año                  174815 non-null  int64  
 3   Cuenca               174815 non-null  str    
 4   Provincia            174815 non-null  str    
 5   Área                 174815 non-null  str    
 6   Yacimiento           174815 non-null  str    
 7   ID Pozo              174815 non-null  int64  
 8   Form.Prod.           174067 non-null  str    
 9   Cód.Propio           161838 non-null  str    
 10  Prod.Men.Pet.(m3)    174815 non-null  float64
 11  Prod.Men.Gas(Mm3)    174815 non-null  float64
 12  Prod.Men.Agua(m3)    174815 non-null  float64
 13  Prod.Acum.Pet.(m3)   174815 non-null  float64
 14  Prod.Acum.Gas(Mm3)   174815 non-null  float64
 15  Prod.Acum.Agua(m3)   174815 

In [5]:
variables_numericas = df.select_dtypes(include="number").columns
variables_categoricas = df.select_dtypes(include="string").columns

tabla_variables = pd.DataFrame({
    "Variables numéricas": pd.Series(variables_numericas),
    "Variables categóricas": pd.Series(variables_categoricas),
})
tabla_variables

,Variables numéricas,Variables categóricas
0,Unnamed: 0,Empresa
1,Año,Cuenca
2,ID Pozo,Provincia
3,Prod.Men.Pet.(m3),Área
4,Prod.Men.Gas(Mm3),Yacimiento
5,Prod.Men.Agua(m3),Form.Prod.
6,Prod.Acum.Pet.(m3),Cód.Propio
7,Prod.Acum.Gas(Mm3),Sist.Extrac.
8,Prod.Acum.Agua(m3),Est.Pozo
9,Iny.Agua(m3),Tipo Pozo


### 1.3 - Valores Faltantes

In [9]:
#Veo las filas para no perderlo de vista
print('\nLa cantidad de filas originales son:')
print(df.shape)

#Calculo faltantes
faltantes = df.isna().sum()
print(faltantes[faltantes > 0])





La cantidad de filas originales son:
(174815, 38)
Form.Prod.                748
Cód.Propio              12977
ETIQUETA               174815
Clasificación           36797
Sub clasificación       36797
Tipo de Recurso         38570
Sub tipo de Recurso    174815
Observaciones          170139
dtype: int64


In [15]:
proporcion_faltantes = faltantes / len(df)

tabla_faltantes = pd.DataFrame({
    "Cantidad de valores faltantes": faltantes,
    "Proporción de valores faltantes (%)": (proporcion_faltantes * 100).round(5),
})
tabla_faltantes[tabla_faltantes["Cantidad de valores faltantes"] > 0]

,Cantidad de valores faltantes,Proporción de valores faltantes (%)
Form.Prod.,748,0.4279
Cód.Propio,12977,7.4233
ETIQUETA,174815,100.0000
Clasificación,36797,21.0491
Sub clasificación,36797,21.0491
Tipo de Recurso,38570,22.0633
Sub tipo de Recurso,174815,100.0000
Observaciones,170139,97.3252


### 1.4 - Variables para caracterizar un pozo

Produccion

RGP

Profundidad

In [16]:
df.head()

,Unnamed: 0,Empresa,Año,Cuenca,Provincia,Área,Yacimiento,ID Pozo,Form.Prod.,Cód.Propio,Prod.Men.Pet.(m3),Prod.Men.Gas(Mm3),Prod.Men.Agua(m3),Prod.Acum.Pet.(m3),Prod.Acum.Gas(Mm3),Prod.Acum.Agua(m3),Iny.Agua(m3),Iny.Gas(Mm3),Iny.CO2(Mm3),Iny.Otros(m3),RGP,ETIQUETA,% de Agua,TEF,Vida Útil,Sist.Extrac.,Est.Pozo,Tipo Pozo,Clasificación,Sub clasificación,Tipo de Recurso,Sub tipo de Recurso,Observaciones,Latitud,Longitud,Cota,Profundidad,Caudal bruta
0,0,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123495,LCAB,201054500100,202.1400,4.0300,878.9500,"490,321.9480","8,455.9770","620,531.0770",0.0000,0.0000,0,0,0.0199,NaN,0.8130,31.0000,"1,356.9800",BCP,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1359,-68.7751,912.0000,"2,317.0000",34.8739
1,1,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123593,CRIN,201054501021,45.1600,0.9000,"2,996.7800","45,706.2050","1,327.2790","559,797.6500",0.0000,0.0000,0,0,0.0199,NaN,0.9852,29.9200,"1,357.3000",BES,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1817,-68.7776,924.0700,"2,330.0000",101.6691
2,2,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123598,CRIN,201054501061,138.0400,2.7400,"4,981.5900","106,582.7870","2,998.7380","923,809.1460",0.0000,0.0000,0,0,0.0198,NaN,0.9730,30.4200,"1,215.5900",BES,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1911,-68.7675,884.5800,"2,450.0000",168.2982
3,3,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123604,CRIN,201054501140,56.2100,1.1200,992.5600,"250,180.0000","7,098.9360","435,091.9350",0.0000,0.0000,0,0,0.0199,NaN,0.9464,31.0000,"1,276.4000",BM,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1822,-68.7972,964.6700,"2,429.0000",33.8313
4,4,YPF,2009,CUYANA,Mendoza,BARRANCAS,BARRANCAS,123612,CRIN,201054501200,233.5900,4.6400,"6,795.1200","319,410.1690","8,596.4720","1,481,590.9560",0.0000,0.0000,0,0,0.0199,NaN,0.9668,31.0000,"1,379.4500",BES,EEF,PET,EXPLOTACION,DESARROLLO,CONVENCIONAL,NaN,NaN,-33.1809,-68.7927,945.0600,"2,374.0000",226.7326


### 1.5 - Tipo de Pozo

Produccion y RGP

## Dataset de entrenamiento y test

# k-NN

# Regresión Logística

# Árboles de Decisión

# Comparación de resultados